In [6]:
import pandas as pd
import numpy as np
import matplotlib.pylab as plt
import seaborn as sns 

pizzas =pd.read_csv("pizzas.csv")
pizza_types = pd.read_csv("pizza_types.csv", encoding='cp1252')
orders = pd.read_csv("orders.csv")
order_details = pd.read_csv("order_details.csv")


In [7]:
tables = {
    'orders': orders,
    'order_details': order_details,
    'pizzas': pizzas,
    'pizza_types': pizza_types,
}

for name, df in tables.items():
    print(f"\n{'='*55}")
    print(f"  TABLE: {name.upper()}")
    print(f"{'='*55}")
    print(f"  Shape      : {df.shape[0]:,} rows  x  {df.shape[1]} cols")
    print(f"  Dtypes:\n{df.dtypes.to_string()}")
    print(f"  Nulls:\n{df.isnull().sum().to_string()}")
    print(f"  Duplicates : {df.duplicated().sum():,}")
    print(f"\n  Sample:\n{df.head(3).to_string()}")


  TABLE: ORDERS
  Shape      : 21,350 rows  x  3 cols
  Dtypes:
order_id     int64
date        object
time        object
  Nulls:
order_id    0
date        0
time        0
  Duplicates : 0

  Sample:
   order_id        date      time
0         1  2015-01-01  11:38:36
1         2  2015-01-01  11:57:40
2         3  2015-01-01  12:12:28

  TABLE: ORDER_DETAILS
  Shape      : 48,620 rows  x  4 cols
  Dtypes:
order_details_id     int64
order_id             int64
pizza_id            object
quantity             int64
  Nulls:
order_details_id    0
order_id            0
pizza_id            0
quantity            0
  Duplicates : 0

  Sample:
   order_details_id  order_id       pizza_id  quantity
0                 1         1     hawaiian_m         1
1                 2         2  classic_dlx_m         1
2                 3         2  five_cheese_l         1

  TABLE: PIZZAS
  Shape      : 96 rows  x  4 cols
  Dtypes:
pizza_id          object
pizza_type_id     object
size              object
pr

In [8]:
# ── Referential Integrity ─────────────────────────────────────────────────────
# السؤال: كل order_id في order_details موجود في orders؟
orphan_orders = order_details[~order_details['order_id'].isin(orders['order_id'])]
print(f"order_details → orders  | Orphan rows: {len(orphan_orders)}")

# السؤال: كل pizza_id في order_details موجود في pizzas؟
orphan_pizzas = order_details[~order_details['pizza_id'].isin(pizzas['pizza_id'])]
print(f"order_details → pizzas  | Orphan rows: {len(orphan_pizzas)}")

# السؤال: كل pizza_type_id في pizzas موجود في pizza_types؟
orphan_types = pizzas[~pizzas['pizza_type_id'].isin(pizza_types['pizza_type_id'])]
print(f"pizzas → pizza_types    | Orphan rows: {len(orphan_types)}")

# ── Value Checks ──────────────────────────────────────────────────────────────
print(f"\nSize values     : {pizzas['size'].unique()}")
print(f"Category values : {pizza_types['category'].unique()}")
print(f"Quantity range  : {order_details['quantity'].min()} → {order_details['quantity'].max()}")
print(f"Price range     : ${pizzas['price'].min()} → ${pizzas['price'].max()}")
print(f"Date range      : {orders['date'].min()} → {orders['date'].max()}")

order_details → orders  | Orphan rows: 0
order_details → pizzas  | Orphan rows: 0
pizzas → pizza_types    | Orphan rows: 0

Size values     : ['S' 'M' 'L' 'XL' 'XXL']
Category values : ['Chicken' 'Classic' 'Supreme' 'Veggie']
Quantity range  : 1 → 4
Price range     : $9.75 → $35.95
Date range      : 2015-01-01 → 2015-12-31


In [11]:
# ── CLEAN: orders ─────────────────────────────────────────────────────────────
# تحويل date و time من نص لأنواع صحيحة
orders['date'] = pd.to_datetime(orders['date'])
orders['time'] = pd.to_datetime(orders['time'], format='%H:%M:%S').dt.time

# ── CLEAN: order_details ──────────────────────────────────────────────────────
# إزالة مسافات زيادة + توحيد الحروف
order_details['pizza_id'] = order_details['pizza_id'].str.strip().str.lower()

# ── CLEAN: pizzas ─────────────────────────────────────────────────────────────
pizzas['pizza_id']      = pizzas['pizza_id'].str.strip().str.lower()
pizzas['pizza_type_id'] = pizzas['pizza_type_id'].str.strip().str.lower()
pizzas['size']          = pizzas['size'].str.strip().str.upper()
pizzas['price']         = pizzas['price'].round(2)

# ── CLEAN: pizza_types ────────────────────────────────────────────────────────
pizza_types['pizza_type_id'] = pizza_types['pizza_type_id'].str.strip().str.lower()
pizza_types['name']          = pizza_types['name'].str.strip()
pizza_types['category']      = pizza_types['category'].str.strip()
pizza_types['ingredients']   = pizza_types['ingredients'].str.strip()

# ── EXPORT ────────────────────────────────────────────────────────────────────
orders.to_csv('orders_clean.csv', index=False)
order_details.to_csv('order_details_clean.csv', index=False)
pizzas.to_csv('pizzas_clean.csv', index=False)
pizza_types.to_csv('pizza_types_clean.csv', index=False)

print("✅ Export done!")

✅ Export done!


In [12]:
from sqlalchemy import create_engine

# ── Connection ────────────────────────────────────────────────────────────────
engine = create_engine(
    "mssql+pyodbc://DESKTOP-VHBF7U2/pizza?driver=ODBC+Driver+17+for+SQL+Server&trusted_connection=yes"
)

# ── Upload ────────────────────────────────────────────────────────────────────
orders.to_sql('orders', engine, if_exists='replace', index=False)
order_details.to_sql('order_details', engine, if_exists='replace', index=False)
pizzas.to_sql('pizzas', engine, if_exists='replace', index=False)
pizza_types.to_sql('pizza_types', engine, if_exists='replace', index=False)

print("✅ Upload done!")

C:\ProgramData\anaconda3\Lib\site-packages\pandas\io\sql.py:1648: SAWarning: Unrecognized server version info '17.0.1000.7'.  Some SQL Server features may not function properly.
  con = self.exit_stack.enter_context(con.connect())


✅ Upload done!
